# PCG Convergence Study  
Using NGSolve's built-in pcg solver and our finite element space hierarchy  


In [ ]:
import numpy as np
from ngsolve import *
from ngsolve.webgui import Draw
import matplotlib.pyplot as plt
from matplotlib import colormaps
import matplotlib.colors as mcolors
import time
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
from src import multigrid_cycles
from multigrid_cycles import *
from src import NGSolve_utils
from NGSolve_utils import *

In [ ]:
# Generating sample of colormaps for plots
mplmaps = list(colormaps)
numcols = len(mplmaps)
color_nums = []
for c in range(numcols):
    amap = plt.get_cmap(mplmaps[c])
    color_nums.append([tuple(amap(x)[:3]) for x in np.linspace(0, 1, 256)])



In [ ]:
# Setting up the problem we will solve on every level

# Boundary conditionis
DIRICHLET = "left|right"

# LHS bilinear form
def poisson_bilinear(a, u, v):
    a += InnerProduct(grad(u), grad(v)) * dx

# RHS linear form
rhs_cf = 0
def poisson_linear(f, u, v):
    f += rhs_cf * v * dx

# Initial iterate
x0 = CoefficientFunction(sin(pi*x)*sin(pi*y)+(1/10)*sin(10*pi*x)*sin(10*pi*y))


In [ ]:

# Call our setup function to put these together
poisson_setup = build_form_setup(bilinear=poisson_bilinear, linear=poisson_linear)

# Define the coarsest mesh for our hierarchy of many sized meshes
N = 16
coarsest_mesh = Mesh(unit_square.GenerateMesh(maxh=1/N))
parfait = build_hierarchy(
    coarsest_mesh,
    poisson_setup,
    n_refines=4,
    order=1,
    dirichlet=DIRICHLET,
    dirichlet_value={"left": 0.0, "right":0.0},
    verbose=True,
)


In [ ]:
finest = parfait.finest
finest.set_initial_guess(x0)

s = Draw(
    finest.gfu,
    finest.mesh,
    "initial guess (finest)",
    deformation=True,
    radius=0.75,
    center=[0.5,0.5,0.5],
    settings={"camera":{"transformations": [{"type": "rotateX", "angle": -45}]}},
)



In [ ]:
# Saving the plots in the different colorways
outdir = Path("colormap_html")
outdir.mkdir(exist_ok=True)

for c in range(numcols):
    current.Redraw(
            finest.gfu,
            finest.mesh,
            colors=color_nums[c],
            colormap_ncolors=256,
            deformation=True,
            radius=0.75,
            center=[0.5,0.5,0.5],
            settings={
                "camera":{"transformations": [
                    {"type": "rotateX", "angle": -45},
                    ]}},
            id=f"Plotted_in_{mplmaps[c]}",
        )
    # current.DownloadScreenshot(f"{mplmaps[c]}.png")